Intent de entrenament d'un model que predigui el millor possible un paràmetre

In [1]:
print("Hola")

Hola


In [2]:
import os
import sys
print(os.getcwd())

sys.path.append(r'c:\Users\jmass\Documents\GitHub\TFG_GEINF\src\data')
from DatasetGenerator import DatasetGenerator


c:\Users\jmass\Documents\GitHub\TFG_GEINF\notebooks\experiments\params_prediction


In [3]:
import pandas as pd

num_particules = 10000

dg = DatasetGenerator(num_particules)

In [4]:
data = dg.generate_BrownianMotion(random_params=True, num_passos=200)

In [5]:
data["attributes"][0]["sigma"]

1.0350206648095814

In [6]:
df = data[["coords", "attributes"]].copy()
df["sigma"] = df["attributes"].apply(lambda x: x["sigma"])
df = df[["coords", "sigma"]]

In [7]:
df

,coords,sigma
0,"[[0.0, -3.0180403265619686, -3.784078504311593...",1.035021
1,"[[0.0, 7.732214479412512, 10.604473522530789, ...",2.440044
2,"[[0.0, -0.5055335286022513, -1.380038372495314...",0.374751
3,"[[0.0, -0.6703714447371922, -0.834253148410375...",0.158770
4,"[[0.0, 1.0324767001450394, -1.6146307031684375...",1.566862
...,...,...
9995,"[[0.0, 3.234413427297062, -3.6496571951040777,...",2.927545
9996,"[[0.0, 1.2481755717346712, 7.00125872806038, 6...",2.100131
9997,"[[0.0, 1.9434916176888866, 2.3190138210573563,...",1.337628
9998,"[[0.0, -0.6087673762739565, 0.2701581645814733...",0.875610


In [8]:
print(df.columns)

Index(['coords', 'sigma'], dtype='object')


In [9]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

In [10]:
# 1. Convertir las coordenadas a tensor 3D
# Asumiendo que 'data' es tu DataFrame
coords_list = [np.stack([row[0], row[1]], axis=1) for row in df['coords']]
X = np.array(coords_list)  # Forma (n_samples, seq_length, 2)
X_tensor = torch.tensor(X, dtype=torch.float32)

In [11]:
# 2. Convertir las etiquetas a tensor de valores reales
max_sigma, min_sigma = df["sigma"].max(), df["sigma"].min()
sigma_series = df["sigma"].apply(lambda x: (x - min_sigma) / (max_sigma - min_sigma))
labels = sigma_series.values  # Suponiendo que 'sigma' contiene los valores reales
y_tensor_temp = torch.tensor(labels, dtype=torch.float32)

In [12]:
y_tensor = y_tensor_temp.unsqueeze(1)  # Agrega una dimensión adicional
y_tensor.shape

torch.Size([10000, 1])

In [13]:
y_tensor

tensor([[0.3222],
        [0.8069],
        [0.0945],
        ...,
        [0.4266],
        [0.2673],
        [0.7855]])

In [14]:
X_tensor = X_tensor.permute(0, 2, 1)  # Nueva forma: (n, 2, m)

In [17]:
# 4. Dividir datos en entrenamiento y prueba (ejemplo simple)
train_size = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:train_size], X_tensor[train_size:]
y_train, y_test = y_tensor[:train_size], y_tensor[train_size:]

In [18]:
# 5. Crear DataLoader
batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [19]:
test_batch_size = 1
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=True)

In [ ]:
# 6. Definir la arquitectura CNN
class TrajectoryCNN(nn.Module):
    def __init__(self, input_channels, min_sigma, max_sigma):
        self.min_sigma = min_sigma
        self.max_sigma = max_sigma
        super().__init__()
        self.first_conv = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv_32_64 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv_64_128 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 4))
        self.fc1 = nn.Linear(4, 256)
        self.fc2 = nn.Linear(256, 1)

    
    def forward(self, data):
        loop = 2
        a_resnet = self.first_conv(data)
        a = self.conv1_32(a_resnet)
        a = self.conv2_32(a)
        a = a + a_resnet
        a = self.max_pool(a)
        b_resnet = self.conv_32_64(a)
        b = self.conv1_64(b_resnet)
        b = self.conv2_64(b)
        b = b + b_resnet
        b = self.max_pool(b)
        c_resnet = self.conv_64_128(b)
        c = self.conv1_128(c_resnet)
        c = self.conv2_128(c)
        c = c + c_resnet
        c = self.avg_pool(c)
        c = c.view(c.size(0), -1)  # Aplanar la salida
        c = F.relu(self.fc1(c))
        c = self.fc2(c)
        c = torch.clamp(c, min=0, max=1)
        return c
    
    def predict_sigma(self, data):
        with torch.no_grad():
            self.eval()
            output = self(data)
            return output.squeeze().numpy() * (self.max_sigma - self.min_sigma) + self.min_sigma
        
    def get_num_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


In [18]:
# import torch
# import torch.nn as nn

# # Parámetros del ejemplo
# input_size = 2       # Por ejemplo, coordenadas x, y
# hidden_size = 64     # Número de neuronas en la capa LSTM
# num_layers = 3       # Capas LSTM apiladas
# output_size = 1      # Valor escalar a predecir

# # Modelo LSTM para regresión
# class LSTMRegressor(nn.Module):
#     def __init__(self):
#         super(LSTMRegressor, self).__init__()
#         self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
#         self.fc = nn.Linear(hidden_size, output_size)

#         self.cnn = nn.Sequential(
#             nn.Conv2d(1, 16, kernel_size=3, padding=1),  # entrada: 1 canal
#             nn.ReLU(),
#             nn.MaxPool2d(2),  # 14x14
#             nn.Conv2d(16, 32, kernel_size=3, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2)   # 7x7
#         )

#     def forward(self, x):
#         out, _ = self.lstm(x)  # out: (batch_size, seq_length, hidden_size)
#         out = out[:, -1, :]    # tomamos la última salida de la secuencia
#         out = self.fc(out)     # regresión final
#         out = torch.clamp(out, min=0, max=1)
#         return out



In [19]:
# # Crear el modelo
# model = LSTMRegressor()

In [21]:
# 7. Inicializar el modelo
model = TrajectoryCNN(
    input_channels=2, min_sigma=0.1, max_sigma=3.0
)

In [22]:
# model.first_conv.weight.shape

In [23]:
# 8. Configurar entrenamiento
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [24]:
import copy

# 9. Entrenar el modelo
num_epochs = 30

best_loss = float('inf')
best_model = copy.deepcopy(model)

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0


    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss/len(train_loader):.4f}")

    # Evaluación
    for x_batch, y_batch in test_loader:
        with torch.no_grad():
            model.eval()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            epoch_loss += loss.item()
    
    if(epoch_loss < best_loss):
        best_loss = epoch_loss
        best_model = copy.deepcopy(model)
    print(f"Epoch {epoch+1}/{num_epochs}, Test Loss: {epoch_loss/len(test_loader):.4f}")

Epoch 1/30, Loss: 0.0551
Epoch 1/30, Test Loss: 0.0573
Epoch 2/30, Loss: 0.0466
Epoch 2/30, Test Loss: 0.0541
Epoch 3/30, Loss: 0.0453
Epoch 3/30, Test Loss: 0.0445
Epoch 4/30, Loss: 0.0447
Epoch 4/30, Test Loss: 0.0462
Epoch 5/30, Loss: 0.0426
Epoch 5/30, Test Loss: 0.0447
Epoch 6/30, Loss: 0.0408
Epoch 6/30, Test Loss: 0.0459
Epoch 7/30, Loss: 0.0408
Epoch 7/30, Test Loss: 0.0423
Epoch 8/30, Loss: 0.0322
Epoch 8/30, Test Loss: 0.0344
Epoch 9/30, Loss: 0.0268
Epoch 9/30, Test Loss: 0.0314
Epoch 10/30, Loss: 0.0262
Epoch 10/30, Test Loss: 0.0295
Epoch 11/30, Loss: 0.0258
Epoch 11/30, Test Loss: 0.0285
Epoch 12/30, Loss: 0.0256
Epoch 12/30, Test Loss: 0.0316
Epoch 13/30, Loss: 0.0265
Epoch 13/30, Test Loss: 0.0295
Epoch 14/30, Loss: 0.0257
Epoch 14/30, Test Loss: 0.0291
Epoch 15/30, Loss: 0.0257
Epoch 15/30, Test Loss: 0.0307
Epoch 16/30, Loss: 0.0256
Epoch 16/30, Test Loss: 0.0372
Epoch 17/30, Loss: 0.0254
Epoch 17/30, Test Loss: 0.0302
Epoch 18/30, Loss: 0.0251
Epoch 18/30, Test Loss:

provar a que el conjunt de train varii a cada epoch

guardar les loss en llistes per poder fer un plot a posteriori


In [27]:
# 10. Valdación (opcional)
with torch.no_grad():
    test_outputs = best_model(X_test).squeeze()  # Asegurarse de que las dimensiones coincidan
    mse = criterion(test_outputs, y_test)  # Calcular el MSE
    print(f'Test MSE: {mse.item():.4f}')


Test MSE: 0.1464


c:\Users\jmass\Documents\GitHub\TFG_GEINF\env\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([2000, 1])) that is different to the input size (torch.Size([2000])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [44]:
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [46]:
# Evaluación
model.eval()  # modo evaluación
total_loss = 0.0
all_preds = []
all_targets = []

In [47]:
with torch.no_grad():  # desactiva cálculo de gradientes
    for x_batch, y_batch in test_loader:
        preds = model(x_batch)
        loss = criterion(preds, y_batch)
        total_loss += loss.item()

        all_preds.append(preds)
        all_targets.append(y_batch)

# Promedio de pérdida
avg_loss = total_loss / len(test_loader)
print(f"Test Loss: {avg_loss:.4f}")


Test Loss: 0.7044


c:\Users\jmass\Documents\GitHub\TFG_GEINF\env\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([16])) that is different to the input size (torch.Size([16, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [ ]:
# Guardar solo los parámetros del modelo
model_info = { # Decidir una estructura
    'model_state_dict': model.state_dict(),
    'input_channels': 2,
    'min_sigma': 0.1,
    'max_sigma': 3.0,
    'model_class': 'TrajectoryCNN'
    
}
torch.save(model_info, 'model_BrownianMotion_sigma_prediction.pth')

# Si deseas guardar el modelo completo
torch.save(model, 'model_BrownianMotion_sigma_prediction_complete.pth')

: 

In [36]:
model.parameters

<bound method Module.parameters of TrajectoryCNN(
  (first_conv): Conv1d(2, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv1_32): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2_32): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv_32_64): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv1_64): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2_64): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv_64_128): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv1_128): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2_128): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (max_pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (avg_pool): AdaptiveAvgPool2d(output_size=(1, 4))
  (fc1): Linear(in_features=4, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=1, bias=True)
)>

In [ ]:
model.__dict__

{'min_sigma': 0.1,
 'max_sigma': 3.0,
 'training': False,
 '_parameters': {},
 '_buffers': {},
 '_non_persistent_buffers_set': set(),
 '_backward_pre_hooks': OrderedDict(),
 '_backward_hooks': OrderedDict(),
 '_is_full_backward_hook': None,
 '_forward_hooks': OrderedDict(),
 '_forward_hooks_with_kwargs': OrderedDict(),
 '_forward_hooks_always_called': OrderedDict(),
 '_forward_pre_hooks': OrderedDict(),
 '_forward_pre_hooks_with_kwargs': OrderedDict(),
 '_state_dict_hooks': OrderedDict(),
 '_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_post_hooks': OrderedDict(),
 '_modules': {'first_conv': Conv1d(2, 32, kernel_size=(3,), stride=(1,), padding=(1,)),
  'conv1_32': Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,)),
  'conv2_32': Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,)),
  'conv_32_64': Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,)),
  'conv1_64': Conv1d(64, 64, kernel_size=(3,), stride=(1,

In [33]:
import inspect

# Extraer la función forward del modelo
forward_code = inspect.getsource(model.forward)
print(forward_code)

    def forward(self, data):
        loop = 2
        a_resnet = self.first_conv(data)
        a = self.conv1_32(a_resnet)
        a = self.conv2_32(a)
        a = a + a_resnet
        a = self.max_pool(a)
        b_resnet = self.conv_32_64(a)
        b = self.conv1_64(b_resnet)
        b = self.conv2_64(b)
        b = b + b_resnet
        b = self.max_pool(b)
        c_resnet = self.conv_64_128(b)
        c = self.conv1_128(c_resnet)
        c = self.conv2_128(c)
        c = c + c_resnet
        c = self.avg_pool(c)
        c = c.view(c.size(0), -1)  # Aplanar la salida
        c = F.relu(self.fc1(c))
        c = self.fc2(c)
        c = torch.clamp(c, min=0, max=1)
        return c



In [ ]:
# Cargar solo los parámetros del modelo
model = TrajectoryCNN(
    input_channels=2,
    seq_length=X_tensor.shape[2],  # m (longitud de la secuencia)
    num_classes=len(le.classes_)
)
model.load_state_dict(torch.load('trajectory_cnn_model_2.pth'))
model.eval()

# Si deseas cargar el modelo completo
# model = torch.load('trajectory_cnn_model_complete.pth')
# model.eval()

In [63]:
model.predict_sigma(torch.tensor(data["coords"][0]).unsqueeze(0))

np.float32(0.35117763)

In [66]:
data["attributes"][0]["sigma"]

2.41768055928194